In [38]:
import pandas as pd, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Conv1D, MaxPooling1D, Dense, Dropout, Flatten
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.optimizers import Adam

PATH        = "data_with_straw.xlsx"
df = (pd.read_excel(PATH)
        .sort_values(["Participant_ID", "Time"])
        .reset_index(drop=True))



In [39]:
print("Columns in file:\n", df.columns.tolist(), "\n")

Columns in file:
 ['Time', 'Zone_0', 'Zone_1', 'Zone_2', 'Zone_3', 'Zone_4', 'Zone_5', 'Zone_6', 'Zone_7', 'Zone_8', 'Zone_9', 'Zone_10', 'Zone_11', 'Zone_12', 'Zone_13', 'Zone_14', 'Zone_15', 'Zone_16', 'Zone_17', 'Zone_18', 'Zone_19', 'Zone_20', 'Zone_21', 'Zone_22', 'Zone_23', 'Zone_24', 'Zone_25', 'Zone_26', 'Zone_27', 'Zone_28', 'Zone_29', 'Zone_30', 'Zone_31', 'Zone_32', 'Zone_33', 'Zone_34', 'Zone_35', 'Zone_36', 'Zone_37', 'Zone_38', 'Zone_39', 'Zone_40', 'Zone_41', 'Zone_42', 'Zone_43', 'Zone_44', 'Zone_45', 'Zone_46', 'Zone_47', 'Zone_48', 'Zone_49', 'Zone_50', 'Zone_51', 'Zone_52', 'Zone_53', 'Zone_54', 'Zone_55', 'Zone_56', 'Zone_57', 'Zone_58', 'Zone_59', 'Zone_60', 'Zone_61', 'Zone_62', 'Zone_63', 'Label', 'Volume', 'Participant_ID', 'Actual_Volume', 'Straw'] 



In [40]:
print("---- Quick stats ------------------------------------------------")
print("Rows                    :", len(df))
print("Participants            :", df['Participant_ID'].nunique())
print("Frames flagged drinking :", (df["Label"]=="Drinking").sum())
print("Frames flagged Not-drinking :", (df["Label"]=="Not_Drinking").sum())
print("Any NaN in ActualVolume :", df['Actual_Volume'].isna().any())
print(df[['Actual_Volume']].describe())

---- Quick stats ------------------------------------------------
Rows                    : 30949
Participants            : 17
Frames flagged drinking : 3057
Frames flagged Not-drinking : 27892
Any NaN in ActualVolume : False
       Actual_Volume
count   30949.000000
mean      228.503451
std       117.811926
min         1.800000
25%       147.000000
50%       197.100000
75%       319.900000
max       511.500000


In [18]:
df 

,Time,Zone_0,Zone_1,Zone_2,Zone_3,Zone_4,Zone_5,Zone_6,Zone_7,Zone_8,...,Zone_59,Zone_60,Zone_61,Zone_62,Zone_63,Label,Volume,Participant_ID,Actual_Volume,Straw
0,1747827431000,284,222,261,317,268,310,2246,2378,321,...,196,178,168,177,201,Not_Drinking,540.8,9,496.190476,Y
1,1747827431200,284,222,248,317,268,310,2246,2337,321,...,201,174,168,179,206,Not_Drinking,540.8,9,496.190476,Y
2,1747827431400,284,222,248,317,268,310,2246,2378,321,...,199,174,168,177,206,Not_Drinking,540.8,9,496.190476,Y
3,1747827431600,284,222,248,317,276,310,2246,2337,321,...,199,174,168,178,208,Not_Drinking,540.8,9,496.190476,Y
4,1747827431800,284,222,248,317,276,310,2246,2337,321,...,199,174,168,178,208,Not_Drinking,540.8,9,496.190476,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30944,1748022490800,217,213,223,230,211,204,234,208,223,...,137,145,159,177,2754,Not_Drinking,15.8,59,3.100000,Y
30945,1748022491000,217,215,223,230,210,204,214,207,212,...,135,140,157,170,203,Not_Drinking,15.8,59,3.100000,Y
30946,1748022491200,207,208,226,228,210,201,214,208,208,...,135,140,155,167,187,Not_Drinking,15.8,59,3.100000,Y
30947,1748022491400,206,203,226,222,202,179,194,204,208,...,133,140,153,167,137,Not_Drinking,15.8,59,3.100000,Y


In [41]:

id_col      = "Participant_ID"
time_col    = "Time"
flag_col    = "Label"           # 'Drinking' / 'Not_Drinking'
vol_col     = "Actual_Volume"

# ────────────────────────────────────────────────────────────────
# 1. LOAD  +  MAP LABEL -> 0/1
# ────────────────────────────────────────────────────────────────


df[flag_col] = df["Label"].map({"Drinking": 1, "Not_Drinking": 0})


# ────────────────────────────────────────────────────────────────
# 2. FIND SIP BOUNDARIES
# ────────────────────────────────────────────────────────────────
df["prev"] = df.groupby(id_col)["Label"].shift(fill_value=0)
df["next"] = df.groupby(id_col)["Label"].shift(-1, fill_value=0)

df["sip_start"] = (df["prev"] == 0) & (df["Label"] == 1)        # first Drinking frame
df["sip_end"]   = (df["Label"] == 1) & (df["next"] == 0)        # last Drinking frame
df["sip_id"]    = df.groupby(id_col)["sip_start"].cumsum()       # 1,2,3,…

# rows where sip_start / sip_end are True
starts = df.loc[df["sip_start"]]
ends   = df.loc[df["sip_end"]]

# ────────────────────────────────────────────────────────────────
# 3.  PICK VOLUME BEFORE-START  &  AFTER-END
#     (guard against boundary rows)
# ────────────────────────────────────────────────────────────────
vol_before = (df.loc[starts.index - 1, vol_col]          # prev row
                .reset_index(drop=True)
                .rename("vol_before"))
vol_after  = (df.loc[ends.index + 1, vol_col]            # next row
                .reset_index(drop=True)
                .rename("vol_after"))

sip_delta = (pd.concat([starts[[id_col, "sip_id"]].reset_index(drop=True),
                        vol_before,
                        vol_after], axis=1)
               .dropna(subset=["vol_before", "vol_after"]))   # drop edges

sip_delta["dV"] = sip_delta["vol_before"] - sip_delta["vol_after"]
print("Non-zero ΔV rows:", (sip_delta["dV"] != 0).sum(), "/", len(sip_delta))
print(sip_delta["dV"].describe())

assert (sip_delta["dV"] != 0).any(), "ΔV still zero – check volume logging!"

# attach dV to every Drinking frame so we can build sequences
df = df.merge(sip_delta[[id_col, "sip_id", "dV"]],
              on=[id_col, "sip_id"], how="left")


Non-zero ΔV rows: 151 / 152
count    152.000000
mean      25.015932
std       15.552482
min        0.000000
25%       13.600000
50%       21.400000
75%       30.025000
max       90.200000
Name: dV, dtype: float64


In [42]:
df

,Time,Zone_0,Zone_1,Zone_2,Zone_3,Zone_4,Zone_5,Zone_6,Zone_7,Zone_8,...,Volume,Participant_ID,Actual_Volume,Straw,prev,next,sip_start,sip_end,sip_id,dV
0,1747827431000,284,222,261,317,268,310,2246,2378,321,...,540.8,9,496.190476,Y,0,0,False,False,0,NaN
1,1747827431200,284,222,248,317,268,310,2246,2337,321,...,540.8,9,496.190476,Y,0,0,False,False,0,NaN
2,1747827431400,284,222,248,317,268,310,2246,2378,321,...,540.8,9,496.190476,Y,0,0,False,False,0,NaN
3,1747827431600,284,222,248,317,276,310,2246,2337,321,...,540.8,9,496.190476,Y,0,0,False,False,0,NaN
4,1747827431800,284,222,248,317,276,310,2246,2337,321,...,540.8,9,496.190476,Y,0,0,False,False,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30944,1748022490800,217,213,223,230,211,204,234,208,223,...,15.8,59,3.100000,Y,0,0,False,False,5,22.5
30945,1748022491000,217,215,223,230,210,204,214,207,212,...,15.8,59,3.100000,Y,0,0,False,False,5,22.5
30946,1748022491200,207,208,226,228,210,201,214,208,208,...,15.8,59,3.100000,Y,0,0,False,False,5,22.5
30947,1748022491400,206,203,226,222,202,179,194,204,208,...,15.8,59,3.100000,Y,0,0,False,False,5,22.5


In [43]:
import pandas as pd
import numpy as np



df['dV'] = df.apply(lambda row: 0 if row['Label'] == 0 or pd.isna(row['dV']) else row['dV'], axis=1)

df.to_excel('dv0_witstraw.xlsx')
print(df)

                Time  Zone_0  Zone_1  Zone_2  Zone_3  Zone_4  Zone_5  Zone_6  \
0      1747827431000     284     222     261     317     268     310    2246   
1      1747827431200     284     222     248     317     268     310    2246   
2      1747827431400     284     222     248     317     268     310    2246   
3      1747827431600     284     222     248     317     276     310    2246   
4      1747827431800     284     222     248     317     276     310    2246   
...              ...     ...     ...     ...     ...     ...     ...     ...   
30944  1748022490800     217     213     223     230     211     204     234   
30945  1748022491000     217     215     223     230     210     204     214   
30946  1748022491200     207     208     226     228     210     201     214   
30947  1748022491400     206     203     226     222     202     179     194   
30948  1748022491600     163     197     178     200     135     150      63   

       Zone_7  Zone_8  ...  Volume  Par

In [47]:


# Assuming df is your original DataFrame
data = df[df['Label'] == 1]
data = data.drop(columns=["Volume"])

# Check for missing values and handle them if necessary
print(data.isnull().sum())
print(data)

# Input features: TOF readings from Zone_0 to Zone_63, and some categorical variables
# Note: Ensure you handle 'sip_id' if used in the input features
X = data.drop(columns=["dV", 'next', 'prev', 'Straw', 'Actual_Volume', 'sip_start', 'sip_end'])
y = data["dV"]  # Target variable

print('X shape:', X.shape)
print('y shape:', y.shape)

# Unique SIP Sessions
unique_sip_ids = data['sip_id'].unique()

# Define the split using 80% of SIP sessions for training
split_index = int(len(unique_sip_ids) * 0.8)
train_sip_ids = unique_sip_ids[:split_index]

# Create training and testing datasets
train_data = data[data['sip_id'].isin(train_sip_ids)]
test_data = data[~data['sip_id'].isin(train_sip_ids)]

# Separate features and target variables
X_train = train_data.drop(columns=["dV", 'next', 'prev', 'Straw', 'Actual_Volume', 'sip_start', 'sip_end'])
y_train = train_data["dV"]

X_test = test_data.drop(columns=["dV", 'next', 'prev', 'Straw', 'Actual_Volume', 'sip_start', 'sip_end'])
y_test = test_data["dV"]
print('Training set shape:', X_train, y_train)
print('Testing set shape:', X_test, y_test)
# Print shapes of the resulting training and testing sets
print('Training set shape:', X_train.shape, y_train.shape)
print('Testing set shape:', X_test.shape, y_test.shape)

# Normalize the input features if desired
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

Time         0
Zone_0       0
Zone_1       0
Zone_2       0
Zone_3       0
            ..
next         0
sip_start    0
sip_end      0
sip_id       0
dV           0
Length: 75, dtype: int64
                Time  Zone_0  Zone_1  Zone_2  Zone_3  Zone_4  Zone_5  Zone_6  \
82     1747827447400     107     100      99     111     134     138     126   
83     1747827447600     107     100      99     111     134     138     126   
84     1747827447800     106     100      98     108     134     136     126   
85     1747827448000     106     103      98     106     129     135     131   
86     1747827448200     115     103      97     103     124     132     132   
...              ...     ...     ...     ...     ...     ...     ...     ...   
30884  1748022478600      82     121     138     135     132     127     113   
30885  1748022478800      81     120     138     135     132     128     122   
30886  1748022479000      79     116     138     136     132     128     122   
30887  174

In [50]:
print(train_data["dV"].describe())

count    2987.000000
mean       27.414090
std        16.830966
min         0.000000
25%        14.666667
50%        22.700000
75%        37.800000
max        90.200000
Name: dV, dtype: float64


In [45]:




# Reshape data for CNN and LSTM
X_train_reshaped = np.reshape(X_train.values, (X_train.shape[0], X_train.shape[1], 1))  # (samples, features, 1)
X_test_reshaped = np.reshape(X_test.values, (X_test.shape[0], X_test.shape[1], 1))  # (samples, features, 1)

# Define input shape for LSTM and CNN
input_shape = (X_train_reshaped.shape[1], 1)  # (timesteps, features)

def build_dnn_model(input_shape):
    model = Sequential()
    model.add(Input(shape=input_shape))  # Use Input layer
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(32, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(1, activation='linear'))  # Regression output
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])
    return model

def build_cnn_model(input_shape):
    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))
    
    model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))
    
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.3))
    
    model.add(Dense(1, activation='linear'))  # Regression output
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])
    return model

def build_lstm_model(input_shape):
    model = Sequential()
    model.add(LSTM(128, input_shape=input_shape, return_sequences=True))
    model.add(Dropout(0.3))
    
    model.add(LSTM(64, return_sequences=False))
    model.add(Dropout(0.3))
    
    model.add(Dense(32, activation='relu'))
    model.add(Dropout(0.3))
    
    model.add(Dense(1, activation='linear'))  # Regression output
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])
    return model

if __name__ == "__main__":
    # # Training DNN
    # dnn_model = build_dnn_model(input_shape=(X_train.shape[1],))  # Correct input shape
    # dnn_model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=1)

    # # Training CNN
    # cnn_model = build_cnn_model(input_shape=input_shape)  # Correct input shape
    # cnn_model.fit(X_train_reshaped, y_train, epochs=100, batch_size=32, verbose=1)

    # Training LSTM
    lstm_model = build_lstm_model(input_shape=(X_train_reshaped.shape[1], 1))  # Correct input shape
    lstm_model.fit(X_train_reshaped, y_train, epochs=100, batch_size=8, verbose=1)

    # Predictions and evaluations follow here...

Epoch 1/100


/opt/anaconda3/envs/ACM1/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


374/374 ━━━━━━━━━━━━━━━━━━━━ 12s 29ms/step - loss: 517.3592 - mae: 17.1435
Epoch 2/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 11s 30ms/step - loss: 303.1083 - mae: 13.3657
Epoch 3/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - loss: 297.6523 - mae: 13.2765
Epoch 4/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 11s 30ms/step - loss: 319.4033 - mae: 13.8070
Epoch 5/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 11s 30ms/step - loss: 324.9514 - mae: 13.7814
Epoch 6/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - loss: 304.1028 - mae: 13.1598
Epoch 7/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 11s 30ms/step - loss: 283.5172 - mae: 12.7670
Epoch 8/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - loss: 261.1588 - mae: 11.8873
Epoch 9/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - loss: 245.5426 - mae: 11.0945
Epoch 10/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - loss: 229.6568 - mae: 10.9447
Epoch 11/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - loss: 227.3853 - mae: 10.6900
Epoch 12/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/st

In [31]:
df

,Time,Zone_0,Zone_1,Zone_2,Zone_3,Zone_4,Zone_5,Zone_6,Zone_7,Zone_8,...,Volume,Participant_ID,Actual_Volume,Straw,prev,next,sip_start,sip_end,sip_id,dV
0,1747827431000,284,222,261,317,268,310,2246,2378,321,...,540.8,9,496.190476,Y,0,0,False,False,0,0.0
1,1747827431200,284,222,248,317,268,310,2246,2337,321,...,540.8,9,496.190476,Y,0,0,False,False,0,0.0
2,1747827431400,284,222,248,317,268,310,2246,2378,321,...,540.8,9,496.190476,Y,0,0,False,False,0,0.0
3,1747827431600,284,222,248,317,276,310,2246,2337,321,...,540.8,9,496.190476,Y,0,0,False,False,0,0.0
4,1747827431800,284,222,248,317,276,310,2246,2337,321,...,540.8,9,496.190476,Y,0,0,False,False,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30944,1748022490800,217,213,223,230,211,204,234,208,223,...,15.8,59,3.100000,Y,0,0,False,False,5,0.0
30945,1748022491000,217,215,223,230,210,204,214,207,212,...,15.8,59,3.100000,Y,0,0,False,False,5,0.0
30946,1748022491200,207,208,226,228,210,201,214,208,208,...,15.8,59,3.100000,Y,0,0,False,False,5,0.0
30947,1748022491400,206,203,226,222,202,179,194,204,208,...,15.8,59,3.100000,Y,0,0,False,False,5,0.0


In [32]:
y_train

82        8.666667
83        8.666667
84        8.666667
85        8.666667
86        8.666667
           ...    
30884    22.500000
30885    22.500000
30886    22.500000
30887    22.500000
30888    22.500000
Name: dV, Length: 2987, dtype: float64

In [33]:
y_test

17487    25.8
17488    25.8
17489    25.8
17490    25.8
17491    25.8
         ... 
17785    22.2
17786    22.2
17787    22.2
17788    22.2
17789    22.2
Name: dV, Length: 70, dtype: float64

In [34]:
X_test

,Time,Zone_0,Zone_1,Zone_2,Zone_3,Zone_4,Zone_5,Zone_6,Zone_7,Zone_8,...,Zone_57,Zone_58,Zone_59,Zone_60,Zone_61,Zone_62,Zone_63,Label,Participant_ID,sip_id
17487,1748003925400,240,250,2692,218,236,235,227,201,238,...,226,221,223,235,268,260,224,1,51,15
17488,1748003925600,240,250,2692,218,236,235,227,201,238,...,226,221,223,235,268,260,224,1,51,15
17489,1748003925800,245,250,253,227,234,235,227,201,247,...,229,220,219,229,255,243,233,1,51,15
17490,1748003926000,245,249,253,227,234,237,225,203,247,...,234,220,219,228,254,242,233,1,51,15
17491,1748003926200,245,248,248,226,236,237,225,205,242,...,234,220,219,228,254,243,224,1,51,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17785,1748003984000,219,213,236,173,216,210,200,189,207,...,213,210,213,215,234,214,210,1,51,18
17786,1748003984200,219,215,232,175,219,211,202,186,208,...,213,207,210,215,234,214,210,1,51,18
17787,1748003984400,219,223,235,177,216,210,202,185,207,...,213,207,210,215,236,208,205,1,51,18
17788,1748003984600,220,215,232,178,216,210,202,185,207,...,213,207,210,218,236,203,205,1,51,18


In [35]:
    for model_name, model, X_eval in [
        # 'DNN',dnn_model, X_test), ('CNN', cnn_model, X_test_reshaped), 
        ('LSTM', lstm_model, X_test_reshaped)]:
        print(f"Evaluating {model_name}...")
        y_pred = model.predict(X_eval)

        # Calculate evaluation metrics
        mse = mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)
        rmspe = np.sqrt(mse) / y_test.mean()
        # Print evaluation metrics
        print(f'{model_name} Evaluation Metrics:')
        print(f'Mean Squared Error (MSE): {mse}')
        print(f'Mean Absolute Error (MAE): {mae}')
        print(f'Root Mean Squared Error (RMSE): {rmse}')
        print(f'R² Score: {r2}')
        print(f"RMSPE {100*np.mean(rmspe):.2f}%")
        # Comparison DataFrame
        comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred.flatten()})
        print(comparison_df.head(50))

Evaluating LSTM...
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
LSTM Evaluation Metrics:
Mean Squared Error (MSE): 61.28624233588537
Mean Absolute Error (MAE): 5.389914937700547
Root Mean Squared Error (RMSE): 7.828553016738494
R² Score: -0.6001424367377035
RMSPE 40.04%
       Actual  Predicted
17487    25.8  24.178278
17488    25.8  24.178278
17489    25.8  24.092339
17490    25.8  24.222557
17491    25.8  24.716183
17492    25.8  24.666899
17493    25.8  24.575233
17494    25.8  23.925766
17495    25.8  24.621218
17496    25.8  23.592640
17497    25.8  24.049156
17498    25.8  23.297482
17499    25.8  23.928024
17500    25.8  24.356277
17590    24.4  26.049185
17591    24.4  26.049185
17592    24.4  26.146429
17593    24.4  26.212296
17594    24.4  23.533615
17595    24.4  23.666630
17596    24.4  23.169952
17597    24.4  23.768869
17598    24.4  24.577208
17599    24.4  23.667202
17600    24.4  23.346245
17676    10.9  25.925974
17677    10.9  25.925974
17678    10.9  26.064814
17679    10

In [36]:
y_pred

array([[24.178278],
       [24.178278],
       [24.092339],
       [24.222557],
       [24.716183],
       [24.666899],
       [24.575233],
       [23.925766],
       [24.621218],
       [23.59264 ],
       [24.049156],
       [23.297482],
       [23.928024],
       [24.356277],
       [26.049185],
       [26.049185],
       [26.14643 ],
       [26.212296],
       [23.533615],
       [23.66663 ],
       [23.169952],
       [23.76887 ],
       [24.577208],
       [23.667202],
       [23.346245],
       [25.925974],
       [25.925974],
       [26.064814],
       [24.62778 ],
       [24.213432],
       [24.288877],
       [24.59598 ],
       [25.04162 ],
       [24.195679],
       [23.729162],
       [23.985123],
       [23.681816],
       [23.32996 ],
       [23.718073],
       [23.548016],
       [23.803226],
       [23.798351],
       [24.113617],
       [23.979795],
       [24.181984],
       [24.133194],
       [24.154346],
       [23.991   ],
       [25.95411 ],
       [25.95411 ],


In [37]:
y_test.head(50)

17487    25.8
17488    25.8
17489    25.8
17490    25.8
17491    25.8
17492    25.8
17493    25.8
17494    25.8
17495    25.8
17496    25.8
17497    25.8
17498    25.8
17499    25.8
17500    25.8
17590    24.4
17591    24.4
17592    24.4
17593    24.4
17594    24.4
17595    24.4
17596    24.4
17597    24.4
17598    24.4
17599    24.4
17600    24.4
17676    10.9
17677    10.9
17678    10.9
17679    10.9
17680    10.9
17681    10.9
17682    10.9
17683    10.9
17684    10.9
17685    10.9
17686    10.9
17687    10.9
17688    10.9
17689    10.9
17690    10.9
17691    10.9
17692    10.9
17693    10.9
17694    10.9
17695    10.9
17696    10.9
17697    10.9
17698    10.9
17768    22.2
17769    22.2
Name: dV, dtype: float64